# Łańcuchy Markowa (poziom znaków)
W tym zeszycie budujemy prosty model językowy oparty na łańcuchach Markowa, który generuje tekst znak po znaku na podstawie częstości występowania par znaków w tekście.

In [ ]:
import random
from collections import Counter
import numpy as np

In [ ]:
# Wczytujemy tekst z pliku
with open('data/pan-tadeusz.txt', 'r', encoding='utf-8') as f:
    content: str = f.read()

print(f"Liczba znaków w tekście: {len(content)}")

In [ ]:
# Zliczamy wystąpienia każdego znaku w tekście
counter: Counter = Counter(content)
print(f"Liczba unikalnych znaków: {len(counter)}")

# Tworzymy listę unikalnych znaków i słownik mapujący znak na jego indeks
chars: list[str] = list(counter.keys())
char2id: dict[str, int] = {c: i for i, c in enumerate(chars)}

In [ ]:
# Inicjalizujemy macierz przejść (liczba wystąpień pary znaków)
n_chars: int = len(chars)
pairs: np.ndarray = np.zeros((n_chars, n_chars))
count: np.ndarray = np.zeros(n_chars)

# Zliczamy przejścia z jednego znaku (c1) do następnego (c2)
for i in range(len(content) - 1):
    c1: str = content[i]
    c2: str = content[i + 1]
    id1: int = char2id[c1]
    id2: int = char2id[c2]
    
    pairs[id1, id2] += 1
    count[id1] += 1

In [ ]:
def generuj_tekst(znak_poczatkowy: str, dlugosc: int = 1000) -> str:
    """
    Generuje tekst używając łańcucha Markowa.
    
    :param znak_poczatkowy: Znak, od którego zaczynamy generowanie tekstu.
    :param dlugosc: Liczba znaków do wygenerowania.
    :return: Wygenerowany ciąg znaków.
    """
    if znak_poczatkowy not in char2id:
        raise ValueError(f"Znak '{znak_poczatkowy}' nie występuje w tekście.")
        
    aktualny_znak: str = znak_poczatkowy
    wynik: list[str] = [aktualny_znak]
    
    for _ in range(dlugosc):
        id_aktualnego: int = char2id[aktualny_znak]
        
        # Obliczamy prawdopodobieństwa przejścia do kolejnych znaków
        # Dzielimy liczbę wystąpień par przez całkowitą liczbę wystąpień aktualnego znaku
        # Zabezpieczenie przed dzieleniem przez zero (np. gdy znak występuje tylko na końcu tekstu)
        if count[id_aktualnego] == 0:
            break
            
        prawdopodobienstwa: np.ndarray = pairs[id_aktualnego] / count[id_aktualnego]
        
        # Losujemy kolejny znak na podstawie wag (prawdopodobieństw)
        aktualny_znak = random.choices(chars, weights=prawdopodobienstwa)[0]
        wynik.append(aktualny_znak)
        
    return "".join(wynik)

In [ ]:
# Generujemy przykładowy tekst zaczynając od litery 'T'
wygenerowany_tekst: str = generuj_tekst('T', dlugosc=500)
print(wygenerowany_tekst)